# Preprocess NHANES wrist accelerometry

`prepare_participant` turns one public NHANES participant into model-ready `day_*.npy` arrays: it downloads the PAX80 archive from the CDC, concatenates the hourly `GT3XPLUS-AccelerationCalibrated-*.sensor.csv` files in time order, and runs the parser in `scripts/get_npy.py` at 80 Hz. One `day_*.npy` is a single calendar day, 00:00 to 24:00 in the monitor’s local time, held as `(2880, 300, 3)` float32: 2880 consecutive 30-second windows of 300 samples at 10 Hz.

Install the project first with `pip install -e .` from the repository root.

Each call returns a `ParticipantResult` with one of three statuses:

- `ok` — eligible days were written
- `no_eligible_day` — the recording was processed, but no day passed quality control
- `failed` — the download or preprocessing raised, and the loop moves on

The archive and CSV are deleted once the arrays exist; `keep_intermediates=True` keeps them. A participant already processed under `WORK_DIR` is read back from disk rather than downloaded again; `overwrite=True` redoes it.

This is a demonstration on three participants. The full cohort belongs in a scheduler job array, with an example shown in the last section.

In [1]:
from pathlib import Path

import pandas as pd

from scripts.tutorial_nhanes.prepare_nhanes_npy import prepare_participant

# WORK_DIR holds <eid>.tar.bz2 and <eid>.csv.gz while a
# participant is being processed, then <eid>/day_*.npy for the arrays.
WORK_DIR = Path('nhanes_data')

# (SEQN, cycle) pairs spanning both NHANES cycles.
PARTICIPANTS = [
    (62161, '2011-2012'),
    (62163, '2011-2012'),
    (73558, '2013-2014'),
]

results = [prepare_participant(eid, cycle, WORK_DIR) for eid, cycle in PARTICIPANTS]
pd.DataFrame(results)[['eid', 'cycle', 'status', 'days_written', 'message']]

,eid,cycle,status,days_written,message
0,62161,2011-2012,ok,6,
1,62163,2011-2012,ok,7,
2,73558,2013-2014,ok,7,


## What the run wrote

Each participant gets one directory. `info.json` records the preprocessing parameters and which days passed. `wear_duration.csv` has one row per calendar day — its wear time, the verdict and the `day_*.npy` it became — which is where the first and last day of a recording usually drop out, since neither spans a full midnight-to-midnight day. Both files are written even for a participant with no eligible day, which has no `day_*.npy` at all. `results[0].load_days()` reads one participant's arrays back.

In [2]:
print(f'{WORK_DIR}/')
for path in sorted(WORK_DIR.rglob('*')):
    indent = '    ' * (len(path.relative_to(WORK_DIR).parts) - 1)
    size = 0 if path.is_dir() else path.stat().st_size
    suffix = '/' if path.is_dir() else f'  {size / 1e6:.1f} MB' if size >= 1e6 else f'  {size / 1e3:.1f} kB'
    print(f'{indent}{path.name}{suffix}')

nhanes_data/
62161/
    day_0.npy  10.4 MB
    day_1.npy  10.4 MB
    day_2.npy  10.4 MB
    day_3.npy  10.4 MB
    day_4.npy  10.4 MB
    day_5.npy  10.4 MB
    info.json  1.7 kB
    wear_duration.csv  0.7 kB
62163/
    day_0.npy  10.4 MB
    day_1.npy  10.4 MB
    day_2.npy  10.4 MB
    day_3.npy  10.4 MB
    day_4.npy  10.4 MB
    day_5.npy  10.4 MB
    day_6.npy  10.4 MB
    info.json  2.0 kB
    wear_duration.csv  0.7 kB
73558/
    day_0.npy  10.4 MB
    day_1.npy  10.4 MB
    day_2.npy  10.4 MB
    day_3.npy  10.4 MB
    day_4.npy  10.4 MB
    day_5.npy  10.4 MB
    day_6.npy  10.4 MB
    info.json  2.0 kB
    wear_duration.csv  0.7 kB


## Scaling to the full cohort

The public cohort holds 14,693 participants, far more than a notebook session can process.

Participants are independent, so `prepare_participant` is also a command-line entry point. How you schedule them depends on your own setup; the minimal Slurm example below runs one participant per array task, reading `SEQN cycle` pairs from a file.

```bash
tail -n +2 scripts/tutorial_nhanes/tabular_files/wearable_ids.csv | cut -d, -f1,2 | tr ',' ' ' > participants.txt
sbatch --array=1-14693%200 --wrap 'read -r SEQN CYCLE < <(sed -n "${SLURM_ARRAY_TASK_ID}p" participants.txt); python -m scripts.tutorial_nhanes.prepare_nhanes_npy --eid "$SEQN" --cycle "$CYCLE" --work-dir /path/to/nhanes'
```

Each task deletes its own archive and CSV, and exits 0 when it wrote eligible days, 3 when no day passed quality control and 1 on failure.